In [ ]:
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
from src.spline_utils import get_bspline_matrix
from src.model import build_credit_model

# --- 1. Term Structure Definition ---
# Using the Malaysian Corporate AAA spread pillars
tenors = np.array([1, 2, 3, 5, 7, 10])
spreads = np.array([85, 102, 120, 155, 180, 210]) 

# --- 2. Out-of-Sample Censoring ---
# We deliberately hide the 5Y observation to test the model's predictive power
tenors_train = np.delete(tenors, 3)
spreads_train = np.delete(spreads, 3)
actual_5y_spread = spreads[3]

# --- 3. Basis Function Generation ---
knots = np.array([1, 3, 5, 7, 10])
X_train = get_bspline_matrix(tenors_train, knots)

# --- 4. Bayesian Posterior Inference ---
credit_model = build_credit_model(X_train, spreads_train)

with credit_model:
    # High target_accept (0.99) eliminates the divergences seen in the terminal
    # Increased tune/draws ensures R-hat stability (< 1.01)
    trace_val = pm.sample(
        draws=2000, 
        tune=2000, 
        target_accept=0.99, 
        random_seed=42,
        progressbar=True
    )